# Quality Vision — Análise Exploratória do Dataset

Este notebook explora o **Casting Product Dataset** antes do treinamento.

Execute após o ETL: `python backend/etl/preprocess.py`

In [ ]:
import json
import random
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from PIL import Image

ROOT = Path('..')
PROCESSED = ROOT / 'data' / 'processed'
META_PATH = PROCESSED / 'metadata.json'

with open(META_PATH) as f:
    meta = json.load(f)

print(f"Total de imagens: {meta['total']}")
print(f"Splits: {meta['splits']}")
print(f"Classes: {meta['class_distribution']}")

In [ ]:
# Distribuição de classes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0a0c10')

# Barplot por split
splits = meta['splits']
axes[0].bar(splits.keys(), splits.values(), color=['#00d4aa', '#ffa502', '#ff4757'])
axes[0].set_title('Imagens por Split', color='white')
axes[0].set_facecolor('#111318')
axes[0].tick_params(colors='white')

# Pie de classes
cd = meta['class_distribution']
axes[1].pie(
    cd.values(), labels=cd.keys(),
    colors=['#00d4aa', '#ff4757'],
    autopct='%1.1f%%', textprops={'color': 'white'}
)
axes[1].set_title('Distribuição de Classes', color='white')
axes[1].set_facecolor('#111318')

plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'class_distribution.png', dpi=120, facecolor='#0a0c10')
plt.show()

In [ ]:
# Visualizar amostras
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Amostras do Dataset — OK (cima) vs Defeito (baixo)', color='white', fontsize=14)
fig.patch.set_facecolor('#0a0c10')

ok_samples = [s for s in meta['samples'] if s['class'] == 'ok' and s['split'] == 'train'][:5]
def_samples = [s for s in meta['samples'] if s['class'] == 'defect' and s['split'] == 'train'][:5]

for i, sample in enumerate(ok_samples):
    img = Image.open(PROCESSED / sample['file'])
    axes[0][i].imshow(img)
    axes[0][i].set_title('OK ✓', color='#00d4aa', fontsize=9)
    axes[0][i].axis('off')

for i, sample in enumerate(def_samples):
    img = Image.open(PROCESSED / sample['file'])
    axes[1][i].imshow(img)
    axes[1][i].set_title('DEFEITO ✗', color='#ff4757', fontsize=9)
    axes[1][i].axis('off')

plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'samples.png', dpi=120, facecolor='#0a0c10')
plt.show()

In [ ]:
# Análise de histograma de pixels
ok_sample = PROCESSED / ok_samples[0]['file']
def_sample = PROCESSED / def_samples[0]['file']

img_ok = np.array(Image.open(ok_sample))
img_def = np.array(Image.open(def_sample))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0a0c10')

for ax, img, title, color in [
    (axes[0], img_ok, 'Peça OK — Histograma RGB', '#00d4aa'),
    (axes[1], img_def, 'Peça com Defeito — Histograma RGB', '#ff4757'),
]:
    ax.set_facecolor('#111318')
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for ch, col in zip(range(3), ['#ff4757', '#00d4aa', '#4dabf7']):
        ax.hist(img[:, :, ch].flatten(), bins=50, alpha=0.6, color=col, label=['R', 'G', 'B'][ch])
    ax.legend()

plt.tight_layout()
plt.show()